In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sb
import skimage
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import scipy.ndimage as ndimage

In [ ]:
# Change dir accordingly:
dir = '/home/caicedo/scr/jcaicedo/Micronuclei-data/'
images_dir = dir + 'dataset_v2/'

files = os.listdir(dir)
annot_files = [x for x in files if x.endswith('csv')]

In [ ]:
# READ DAPI IMAGE
def read_dapi_image(imid):
    # 20X_c0-DAPI_A1_Tile-110.phenotype.tif
    mag, well, tile = imid.split('_')
    imname = f'{mag}_c0-DAPI_{well}_{tile}.phenotype.tif'
    im = skimage.io.imread(images_dir + imname)
    return im

In [ ]:
# READ MICRONUCLEI ANNOTATIONS
def read_micronuclei_annotations(imid):
    # 20X_c0-DAPI_A1_Tile-130.phenotype_outlines.png
    mag, well, tile = imid.split('_')
    otlname = f'{mag}_c0-DAPI_{well}_{tile}.phenotype_outlines.png'
    otl = skimage.io.imread(images_dir + otlname)
    otl = otl[:,:,0] > 0 # Only red channel and binarize
    
    otl = ndimage.binary_fill_holes(otl) ^ otl
    labels = skimage.measure.label(otl)

    return labels

In [ ]:
def get_micronuclei_locations(labels):
    data = []
    for i in range(1,len(np.unique(labels))):
        ys,xs = np.where(labels == i)
        a,b = int(np.mean(ys)), int(np.mean(xs))
        area = np.sum(labels == i)
        data.append({"x":b, "y":a, "area":area})

    mni = pd.DataFrame(data=data, columns=["x","y","area"])
    return mni

In [ ]:
def display_data(im, mni, df):
    # Show image
    fig, ax = plt.subplots(figsize=(30,30))
    ax.imshow(im)

    # Display nucleus boxes
    w,h = 32,32
    for k,r in df.iterrows():
        x = int(r.j_nucleus)
        y = int(r.i_nucleus)
        x1 = max(0, x - w)
        x2 = min(x + w, im.shape[1])
        y1 = max(0, y - h)
        y2 = min(y + h, im.shape[0])
        # Create a Rectangle patch
        rect = patches.Rectangle((x1, y1), 2*w, 2*h, linewidth=1, edgecolor='c', facecolor='none')
        ax.add_patch(rect)

    # Display micronucleus boxes
    w,h = 16,16
    for k,r in mni.iterrows():
        x1 = r.x - w
        y1 = r.y - h
        rect = patches.Rectangle((x1, y1), 2*w, 2*h, linewidth=2, edgecolor='r', facecolor='none')
        ax.add_patch(rect)

    #plt.axis('off')
    plt.show()


In [ ]:
count_annotations = 0
mni_dfs = []
for fname in annot_files:
    imid = fname.split('.')[0]
    im = read_dapi_image(imid)
    labels = read_micronuclei_annotations(imid)
    
    mni = get_micronuclei_locations(labels)
    count_annotations += len(mni)
    
    df = pd.read_csv(dir + fname)
    print(f"{imid}: micronuclei:{len(mni)}")
    display_data(im, mni, df)
    
    mni["Image"] = imid
    mni_dfs.append(mni)
    
print("Total micronuclei:",count_annotations)

In [ ]:
MNI = pd.concat(mni_dfs)

In [ ]:
bins = [x for x in range(0,100,2)]
sb.histplot(data=MNI, x="area", bins=bins)

In [ ]:
bins = [x for x in range(0,800,10)]
sb.histplot(data=MNI, x="area", bins=bins)

In [ ]:
patch = skimage.exposure.rescale_intensity(im, out_range=np.float32)
sobel = skimage.filters.sobel(patch)
sobel = 2*skimage.exposure.rescale_intensity(sobel, out_range=np.float32)
sobel[sobel > 1] = 1
px = np.concatenate(
    (sobel[:,:,np.newaxis], patch[:,:,np.newaxis], patch[:,:,np.newaxis]), 
    axis=2)

In [ ]:
plt.imshow(px[750:1000,100:350])